<a href="https://colab.research.google.com/github/maewhyxng2-prog/zyrquen-platform/blob/main/experiments_native_training_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZYRQUEN Native V2 — Checkpoint Reload & Continuation Lab

**สถานะปัจจุบัน:** `CANDIDATE — NOT PROMOTED`  
**โมเดล:** `PureNativeTransformerLMV2` + `PureContinualTrainerV2` (Native Python Core, ไม่ใช้ pretrained weights ภายนอก และไม่เรียก AI API ภายนอก)

### ลำดับ Checkpoint ล่าสุดในระบบ
| Stage | File | Global Step | SHA-256 |
|---|---|---:|---|
| Historical Baseline | `v2_continued_step4000.json.gz` | 4000 | `0ca27a632e85bc7fde764b9948ceb9ed16a53911d5773bd51b5cd7c722a5237d` |
| Historical Continuation | `v2_continued_step4025.json.gz` | 4025 | `f4b33c82a27a0a3f05b468d2ff1cf297746f7a3aa85ecbc03ff0093f34c75314` |
| Step 5000 Baseline | `v2_teacher_transfer_step5000.json.gz` | 5000 | `76c85f56c615590906bee1f7893e22301f7be1d430130cd2b882e1921d9d72ed` |
| **Latest Candidate** | **`v2_teacher_transfer_run5000_step6000.json.gz`** | **6000** | **`3b7f0ed4de259a0b1db2135caf193c074409cadb3a599ae4656cfd4b3dffbca7`** |

> **หมายเหตุสำคัญ:** ไฟล์ checkpoint (`.json.gz`) ถูกเก็บไว้ใน Colab / Cloud Storage (`/content/zyrquen-resume/checkpoints`) ไม่ได้ commit ลง GitHub  
> โน้ตบุ๊กนี้จะตรวจสอบ SHA-256 และค้นหา checkpoint ขั้นสูงสุดที่มีอยู่จริงใน `OUT_DIR` โดยอัตโนมัติ (เลือก `v2_teacher_transfer_run5000_step6000.json.gz` ก่อน และรองรับ fallback ไปยัง `step5000` หรือ `step4025`)

In [ ]:
#@title 1. ตั้งค่าสภาพแวดล้อม (Environment & Repository Setup)
import os
import subprocess
import sys
from pathlib import Path

# ปรับ path ตามสภาพแวดล้อม Colab / Local ของคุณ
REPO_DIR = Path("/content/zyrquen-platform")
OUT_DIR = Path("/content/zyrquen-resume/checkpoints")
REPO_URL = "https://github.com/maewhyxng2-prog/zyrquen-platform.git"

# (ตัวเลือก) Mount Google Drive หากเก็บ checkpoint หรือ dataset ไว้ใน Drive
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
if USE_GOOGLE_DRIVE and "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

if not REPO_DIR.exists():
    if (Path.cwd() / "python_core" / "v2" / "zyrquen_v2_pure.py").exists():
        REPO_DIR = Path.cwd()
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

PYTHON_CORE = REPO_DIR / "python_core"
for p in (str(PYTHON_CORE), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from tools.resume_v2_lab import (
    KNOWN_CHECKPOINT_SHA256,
    find_latest_checkpoint,
    load_texts_from_file,
    resolve_expected_sha256,
    sha256_file,
    verify_and_reload,
)
from v2.zyrquen_v2_pure import PureContinualTrainerV2
from v2.zyrquen_v2_trainer import ByteTokenizer
from verify_checkpoints import verify_checkpoints, print_report

print(f"Python   : {sys.version.split()[0]}")
print(f"REPO_DIR : {REPO_DIR}")
print(f"OUT_DIR  : {OUT_DIR} (exists={OUT_DIR.exists()})")

In [ ]:
#@title 2. ตรวจ SHA-256 ของ Checkpoint ทั้งหมด และเลือก Checkpoint ล่าสุดอัตโนมัติ
assert OUT_DIR.is_dir(), f"ไม่พบโฟลเดอร์ checkpoint: {OUT_DIR}"

# ตรวจความถูกต้องของทุกไฟล์ใน OUT_DIR เทียบกับ manifest ที่บันทึกไว้
overrides = {
    fname: digest
    for fname, digest in KNOWN_CHECKPOINT_SHA256.items()
    if (OUT_DIR / fname).exists()
}
report = verify_checkpoints(OUT_DIR, expected_overrides=overrides, allow_empty=False)
print_report(report)
assert report["ok"], "พบ checkpoint ที่ SHA-256 ไม่ตรงหรือมีข้อผิดพลาด กรุณาตรวจสอบก่อนฝึกต่อ"

# เลือก checkpoint ที่มี step สูงสุด (เช่น v2_teacher_transfer_run5000_step6000.json.gz)
LATEST_CKPT, LATEST_SHA = find_latest_checkpoint(OUT_DIR)
BASELINE_5000 = OUT_DIR / "v2_teacher_transfer_step5000.json.gz"
BASELINE_5000_SHA = KNOWN_CHECKPOINT_SHA256["v2_teacher_transfer_step5000.json.gz"]

print(f"\nSelected latest checkpoint : {LATEST_CKPT.name}")
print(f"Verified SHA-256           : {sha256_file(LATEST_CKPT)}")
if BASELINE_5000.is_file():
    assert sha256_file(BASELINE_5000) == BASELINE_5000_SHA, "Step 5000 baseline SHA-256 mismatch"
    print(f"Step 5000 baseline OK      : {BASELINE_5000.name}")

In [ ]:
#@title 3. ทดสอบ Fresh-Runtime Reload & ตรวจสอบสถานะ Optimizer / Continual Learning
reloaded = verify_and_reload(LATEST_CKPT, expected_sha256=LATEST_SHA)
trainer: PureContinualTrainerV2 = reloaded["trainer"]
tokenizer: ByteTokenizer = reloaded["tokenizer"]

print("=== Reload Verification Summary ===")
print(f"Checkpoint          : {reloaded['checkpoint']}")
print(f"Size (bytes)        : {reloaded['size_bytes']:,}")
print(f"Embedded global_step: {reloaded['global_step']}")
print(f"Replay buffer items : {reloaded['replay_buffer_len']}")
print(f"Anchor params       : {reloaded['has_anchor_params']}")
print(f"Importance diag     : {reloaded['has_importance_diag']}")

# (แนะนำ) ประเมินผลเทียบ Step 5000 baseline บนชุด held-out 4 ตัวอย่างเดียวกันก่อนฝึกต่อ
# ระบุ path ของไฟล์ held-out validation (JSON/JSONL) หรือกำหนด list ของข้อความโดยตรง
VAL_FILE = None  # ตัวอย่าง: Path("/content/zyrquen-resume/data/transfer_val.jsonl")
held_out_texts = load_texts_from_file(VAL_FILE) if VAL_FILE else []

if held_out_texts:
    cand_metrics = trainer.model.evaluate_texts(tokenizer, held_out_texts)
    print(f"\nCandidate (step {trainer.global_step}) held-out metrics: {cand_metrics}")
    if BASELINE_5000.is_file():
        base_reloaded = verify_and_reload(BASELINE_5000, expected_sha256=BASELINE_5000_SHA)
        base_metrics = base_reloaded["trainer"].model.evaluate_texts(tokenizer, held_out_texts)
        print(f"Baseline  (step 5000) held-out metrics: {base_metrics}")
else:
    print("\n[INFO] ยังไม่ได้กำหนด VAL_FILE — ข้ามการประเมิน held-out (ไม่สร้างคะแนนจำลอง)")

In [ ]:
#@title 4. ฝึกต่อจาก Checkpoint ล่าสุด (Continuation Training Loop)
# กำหนดชุดข้อมูลฝึก (16 ตัวอย่างจาก teacher transfer หรือชุดใหม่ที่แยก validation split แล้ว)
TRAIN_FILE = None  # ตัวอย่าง: Path("/content/zyrquen-resume/data/transfer_train.jsonl")
train_texts = load_texts_from_file(TRAIN_FILE) if TRAIN_FILE else []

TARGET_STEP = 7000  #@param {type:"integer"}
CHUNK_STEPS = 25    #@param {type:"integer"}
BATCH_SIZE = 3      #@param {type:"integer"}
REPLAY_RATIO = 0.5  #@param {type:"number"}

if not train_texts:
    print("[WAIT] กรุณากำหนด TRAIN_FILE หรือใส่รายการข้อความใน train_texts ก่อนเริ่มฝึกต่อ")
else:
    start_step = trainer.global_step
    run_prefix = f"v2_teacher_transfer_run{start_step}"
    print(f"Starting continuation: step {start_step} -> {TARGET_STEP} | examples={len(train_texts)}")

    while trainer.global_step < TARGET_STEP:
        step_budget = min(CHUNK_STEPS, TARGET_STEP - trainer.global_step)
        stats = trainer.train_steps(
            train_texts,
            steps=step_budget,
            batch_size=BATCH_SIZE,
            replay_ratio=REPLAY_RATIO,
        )
        out_ckpt = OUT_DIR / f"{run_prefix}_step{trainer.global_step}.json.gz"
        _, digest = trainer.save_checkpoint(out_ckpt)
        print(
            f"STEP {trainer.global_step} | loss={stats['final_loss']:.4f} | "
            f"file={out_ckpt.name} | sha256={digest}"
        )

In [ ]:
#@title 5. ตรวจสอบความสมบูรณ์หลังฝึกจบ และประเมิน Held-Out (Post-Run Gate)
post_report = verify_checkpoints(OUT_DIR, allow_empty=False)
print(f"Directory check: total={post_report['total']} matched={post_report['matched']} failed={post_report['failed']}")

if held_out_texts:
    after_metrics = trainer.model.evaluate_texts(tokenizer, held_out_texts)
    print(f"Post-training held-out (step {trainer.global_step}): {after_metrics}")

# ตัวอย่าง CLI แบบบรรทัดเดียวสำหรับรันจาก Terminal หรือ Colab shell:
# !python3 tools/resume_v2_lab.py --out-dir /content/zyrquen-resume/checkpoints --verify-only